# 03 Extracting features

Here I turn photos and descriptions into feature vectors using pretrained models that aren't retrained (frozen features). Our dataset is fairly small for training image or text models from scratch, and Week 5 noted CNNs are the better choice when data is limited.

I extract three image versions so I can compare them: EfficientNet-B0 with its default resize and centre crop (what our proposal planned), EfficientNet-B0 with padding instead of cropping, and ResNet-50 with padding as a second CNN. Text uses BERT with mean pooling. I also measure photo quality and how long each model takes per item.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# same folder setup as the other notebooks
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/airbnb_p22")
except ImportError:
    PROJECT_DIR = Path("airbnb_p22")

DATA_DIR = PROJECT_DIR / "data"
PHOTO_DIR = DATA_DIR / "photos_480"
FEATURE_DIR = DATA_DIR / "features"
RESULTS_DIR = PROJECT_DIR / "results"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (switch to a T4!)")

try:
    df = pd.read_parquet(DATA_DIR / "listings_clean.parquet")
except OSError as e:
    raise RuntimeError(f"Run notebook 02 first: {e}")
# every feature file uses this same row order, so row i is always the same listing
row_of = pd.Series(np.arange(len(df)), index=df["id"])
print(f"{len(df):,} listings, {df['has_photo'].sum():,} with a photo")

Mounted at /content/drive
Using: Tesla T4
18,808 listings, 18,607 with a photo


In [ ]:
import zipfile

# checking every photo zip is there and readable, because the first run
# only got through 5 of them
files = sorted(PHOTO_DIR.glob("photos_*.zip"))
print(f"{len(files)} zip files found")
total = 0
for p in files:
    try:
        with zipfile.ZipFile(p) as z:
            n = sum(name.endswith(".jpg") for name in z.namelist())
        total += n
        print(f"{p.name}: {p.stat().st_size / 1e6:.0f} MB, {n} photos")
    except (zipfile.BadZipFile, OSError) as e:
        print(f"{p.name}: ERROR {e}")
print(f"Total photos: {total:,}")
# 19,004 listings in shards of 1,000 means 20 zips; stop now if any are missing
assert len(files) == 20, f"Only {len(files)} of 20 photo zips found, check Drive first"

20 zip files found
photos_000.zip: 45 MB, 994 photos
photos_001.zip: 43 MB, 994 photos
photos_002.zip: 43 MB, 993 photos
photos_003.zip: 44 MB, 997 photos
photos_004.zip: 43 MB, 997 photos
photos_005.zip: 43 MB, 993 photos
photos_006.zip: 43 MB, 992 photos
photos_007.zip: 42 MB, 994 photos
photos_008.zip: 42 MB, 991 photos
photos_009.zip: 42 MB, 997 photos
photos_010.zip: 41 MB, 992 photos
photos_011.zip: 41 MB, 992 photos
photos_012.zip: 41 MB, 996 photos
photos_013.zip: 42 MB, 991 photos
photos_014.zip: 42 MB, 984 photos
photos_015.zip: 42 MB, 986 photos
photos_016.zip: 41 MB, 981 photos
photos_017.zip: 41 MB, 982 photos
photos_018.zip: 40 MB, 952 photos
photos_019.zip: 0 MB, 4 photos
Total photos: 18,802


## Padding and photo quality

Most photos are 3:2 landscape. The default EfficientNet preprocessing resizes the short side to 256 and cuts a 224 x 224 square from the middle, which keeps only about half of a 3:2 photo. Padding shrinks the whole photo to fit 224 pixels on its longest side and fills the gap with the ImageNet average colour, so the padding becomes about zero after normalising and nothing gets stretched (Week 2 demo).

For quality I use the Week 2 checks: Laplacian variance for blur, mean brightness, and the 2nd to 98th percentile range for contrast, all measured on the 480 pixel copy.

In [ ]:
import io
import zipfile

import cv2
from PIL import Image

IMG_SIZE = 224
# ImageNet mean colour, so after normalising the padding comes out as roughly 0
PAD_COLOUR = (124, 116, 104)


def letterbox(img, size=IMG_SIZE):
    # shrink (or enlarge) so the longest side fits, then pad the rest,
    # so the whole photo is kept and nothing gets stretched
    scale = size / max(img.size)
    new_size = (max(1, round(img.width * scale)), max(1, round(img.height * scale)))
    img = img.resize(new_size, Image.BICUBIC)
    canvas = Image.new("RGB", (size, size), PAD_COLOUR)
    canvas.paste(img, ((size - new_size[0]) // 2, (size - new_size[1]) // 2))
    return canvas


def photo_quality(img):
    # Week 2 checks: Laplacian variance for blur, mean for brightness,
    # 2nd to 98th percentile range for contrast. All on the 480px copy.
    gray = np.asarray(img.convert("L"))
    p2, p98 = np.percentile(gray, [2, 98])
    return {"blur": cv2.Laplacian(gray, cv2.CV_64F).var(), "brightness": gray.mean(),
            "contrast": p98 - p2, "short_side": min(img.size)}


def iter_photos(verbose=False):
    # goes through the zip files one at a time and only returns photos
    # for listings that are in our clean table. One bad photo only skips
    # that photo, not the rest of its zip file.
    for path in sorted(PHOTO_DIR.glob("photos_*.zip")):
        n_ok, n_bad = 0, 0
        try:
            with zipfile.ZipFile(path) as z:
                for name in z.namelist():
                    if not name.endswith(".jpg"):
                        continue
                    listing_id = int(name.split(".")[0])
                    if listing_id not in row_of.index:
                        continue
                    try:
                        img = Image.open(io.BytesIO(z.read(name))).convert("RGB")
                    except Exception:
                        n_bad += 1
                        continue
                    n_ok += 1
                    yield listing_id, img
        except (zipfile.BadZipFile, OSError) as e:
            print(f"Problem reading {path.name}: {e}")
        if verbose:
            print(f"{path.name}: {n_ok} photos read, {n_bad} failed")

## Image features

Both models are pretrained on ImageNet and used without retraining. I remove their final classification layer and keep the features just before it: 1,280 numbers per photo for EfficientNet-B0 and 2,048 for ResNet-50. Listings without a photo get a row of zeros, and the has_photo column tells the models later which rows those are.

In [ ]:
import torchvision
from torchvision import transforms

normalise = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eff_weights = torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1
res_weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2

# Three image feature sets:
#  - EfficientNet-B0 with its default preprocessing (resize, then centre crop)
#  - EfficientNet-B0 with padding instead of cropping
#  - ResNet-50 with padding, to compare a second CNN on the same input
eff = torchvision.models.efficientnet_b0(weights=eff_weights)
res = torchvision.models.resnet50(weights=res_weights)
eff.classifier = torch.nn.Identity()   # take the 1,280 features before the classifier
res.fc = torch.nn.Identity()           # and the 2,048 features for ResNet
crop_input = eff_weights.transforms()   # resize to 256, centre crop 224, normalise
dims = {"effnet_crop": 1280, "effnet_pad": 1280, "resnet_pad": 2048}
for model in (eff, res):
    model.eval().to(device)

features = {name: np.zeros((len(df), d), dtype=np.float32) for name, d in dims.items()}
gpu_seconds = dict.fromkeys(dims, 0.0)
quality_rows = []
BATCH = 128


@torch.no_grad()
def run_batch(ids, imgs):
    rows = row_of[ids].to_numpy()
    x_crop = torch.stack([crop_input(im) for im in imgs]).to(device)
    # the padded version is made once and used by both models
    x_pad = torch.stack([normalise(letterbox(im)) for im in imgs]).to(device)
    for name, model, x in [("effnet_crop", eff, x_crop), ("effnet_pad", eff, x_pad),
                           ("resnet_pad", res, x_pad)]:
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        out = model(x)
        if device == "cuda":
            torch.cuda.synchronize()   # wait for the GPU so the timing is real
        gpu_seconds[name] += time.time() - t0
        features[name][rows] = out.cpu().numpy()


start = time.time()
batch_ids, batch_imgs = [], []
for listing_id, img in iter_photos(verbose=True):
    quality_rows.append({"id": listing_id, **photo_quality(img)})
    batch_ids.append(listing_id)
    batch_imgs.append(img)
    if len(batch_ids) == BATCH:
        run_batch(batch_ids, batch_imgs)
        batch_ids, batch_imgs = [], []
        if len(quality_rows) % 2560 == 0:
            print(f"{len(quality_rows):,} photos done")
if batch_ids:
    run_batch(batch_ids, batch_imgs)
n_photos = len(quality_rows)
print(f"{n_photos:,} photos in {(time.time() - start) / 60:.1f} min")
# stop here if photos went missing again, so bad features never get saved
expected = df["has_photo"].sum()
assert n_photos == expected, f"Only {n_photos:,} of {expected:,} photos were processed"

# listings without a photo keep a row of zeros; has_photo tells the models which ones
try:
    for name, arr in features.items():
        np.save(FEATURE_DIR / f"image_{name}.npy", arr)
    np.save(FEATURE_DIR / "ids.npy", df["id"].to_numpy())
    pd.DataFrame(quality_rows).to_csv(RESULTS_DIR / "photo_quality.csv", index=False)
except OSError as e:
    raise RuntimeError(f"Couldn't save the image features: {e}")
for name in dims:
    print(f"{name}: {1000 * gpu_seconds[name] / n_photos:.2f} ms per photo on the GPU")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 133MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 112MB/s]


photos_000.zip: 980 photos read, 0 failed
photos_001.zip: 979 photos read, 0 failed
2,560 photos done
photos_002.zip: 983 photos read, 0 failed
photos_003.zip: 986 photos read, 0 failed
photos_004.zip: 983 photos read, 0 failed
5,120 photos done
photos_005.zip: 983 photos read, 0 failed
photos_006.zip: 981 photos read, 0 failed
7,680 photos done
photos_007.zip: 989 photos read, 0 failed
photos_008.zip: 982 photos read, 0 failed
photos_009.zip: 991 photos read, 0 failed
10,240 photos done
photos_010.zip: 987 photos read, 0 failed
photos_011.zip: 983 photos read, 0 failed
photos_012.zip: 988 photos read, 0 failed
12,800 photos done
photos_013.zip: 982 photos read, 0 failed
photos_014.zip: 972 photos read, 0 failed
15,360 photos done
photos_015.zip: 978 photos read, 0 failed
photos_016.zip: 970 photos read, 0 failed
photos_017.zip: 975 photos read, 0 failed
17,920 photos done
photos_018.zip: 932 photos read, 0 failed
photos_019.zip: 3 photos read, 0 failed
18,607 photos in 6.1 min
effnet_

## Photo quality results

The blur threshold of 100 is the one from Week 2. The brightness and contrast cut-offs are my own rough thresholds, just to flag extreme photos. I also check whether quality differs by price level: if premium photos are simply sharper or brighter, the image model could be picking up photo quality rather than the property itself.

In [ ]:
import matplotlib.pyplot as plt

q = pd.DataFrame(quality_rows)
print(f"Blurry (Laplacian variance under 100): {(q['blur'] < 100).mean():.1%}")
print(f"Very dark (mean brightness under 50): {(q['brightness'] < 50).mean():.1%}")
print(f"Low contrast (p2 to p98 range under 100): {(q['contrast'] < 100).mean():.1%}")
print(f"Small (short side under 224px, gets enlarged): {(q['short_side'] < IMG_SIZE).mean():.1%}")

# does photo quality differ by price level? if premium photos are just sharper,
# the image model might be learning photo quality rather than the property
q = q.merge(df[["id", "price_level"]], on="id")
print(q.groupby("price_level", observed=True)[["blur", "brightness", "contrast"]]
      .median().round(1))

# what padding looks like compared with the default crop
sample = next(im for _, im in iter_photos() if im.width / im.height > 1.4)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
crop_view = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])(sample)
for ax, im, title in zip(axes, [sample, crop_view, letterbox(sample)],
                         ["Original", "Default: resize + centre crop", "Padding"]):
    ax.imshow(im)
    ax.set_title(title)
    ax.axis("off")
plt.suptitle("What the image model sees")
plt.tight_layout()
plt.show()

Blurry (Laplacian variance under 100): 0.2%
Very dark (mean brightness under 50): 0.5%
Low contrast (p2 to p98 range under 100): 0.1%
Small (short side under 224px, gets enlarged): 1.5%
               blur  brightness  contrast
price_level                              
budget        787.4       135.2     209.0
mid-range    1014.2       146.8     210.0
premium      1135.2       144.6     208.0


[Photo grid removed before publishing: the photos belong to the hosts. Run the cell to see it.]


## Text features

BERT (bert-base-uncased) with mean pooling, as planned in the proposal and used in the Week 4 practical. Mean pooling averages the last layer over the real tokens, with padding masked out. Before choosing a 256 token limit I check how many descriptions would get cut off. Empty descriptions get a row of zeros, and has_description marks them.

In [ ]:
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased").eval().to(device)
MAX_LEN = 256

texts = df["description_clean"].fillna("").tolist()
# check how many descriptions are longer than BERT's input limit before choosing it
lengths = np.array([len(ids) for ids in tokenizer(texts)["input_ids"]])
print(f"Tokens per description: median {np.median(lengths):.0f}, max {lengths.max()}")
print(f"Cut off at {MAX_LEN} tokens: {(lengths > MAX_LEN).mean():.2%}")

text_features = np.zeros((len(df), bert.config.hidden_size), dtype=np.float32)
has_text = df["has_description"].to_numpy()
rows = np.where(has_text)[0]
start = time.time()
with torch.no_grad():
    for i in range(0, len(rows), 64):
        idx = rows[i:i + 64]
        enc = tokenizer([texts[j] for j in idx], padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt").to(device)
        hidden = bert(**enc).last_hidden_state
        # mean pooling over the real tokens only, the padding is masked out
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden * mask).sum(1) / mask.sum(1)
        text_features[idx] = pooled.cpu().numpy()
        if (i // 64) % 50 == 0:
            print(f"{i:,} of {len(rows):,} descriptions done")

if device == "cuda":
    torch.cuda.synchronize()
bert_seconds = time.time() - start
print(f"BERT: {len(rows):,} descriptions in {bert_seconds / 60:.1f} min, "
      f"{1000 * bert_seconds / len(rows):.2f} ms each")

# empty descriptions keep a row of zeros, has_description marks them
try:
    np.save(FEATURE_DIR / "text_bert.npy", text_features)
    timing = {f"image_{k}": 1000 * v / n_photos for k, v in gpu_seconds.items()}
    timing["text_bert"] = 1000 * bert_seconds / len(rows)
    pd.Series(timing, name="ms_per_item").to_csv(RESULTS_DIR / "feature_timing.csv")
except OSError as e:
    raise RuntimeError(f"Couldn't save the text features: {e}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens per description: median 94, max 235
Cut off at 256 tokens: 0.00%
0 of 18,737 descriptions done
3,200 of 18,737 descriptions done
6,400 of 18,737 descriptions done
9,600 of 18,737 descriptions done
12,800 of 18,737 descriptions done
16,000 of 18,737 descriptions done
BERT: 18,737 descriptions in 2.8 min, 8.82 ms each


## Which features to carry forward

Same re-measure idea as Week 2: I train the same simple classifier (scaled features, logistic regression) on each feature set and score it on validation only. This is just for choosing features (crop or padding, EfficientNet or ResNet, BERT or TF-IDF). The proper models come in notebook 04, and the test set still isn't touched.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Quick check like the Week 2 re-measure: the same simple classifier on each
# feature set, scored on validation. It's only for choosing which features to
# carry forward, the proper models come in notebook 04.
y = df["price_level"].cat.codes.to_numpy()
is_train = (df["split"] == "train").to_numpy()
is_val = (df["split"] == "val").to_numpy()
photo = df["has_photo"].to_numpy()

candidates = {name: (arr, photo) for name, arr in features.items()}
candidates["text_bert"] = (text_features, has_text)
rows = []
for name, (X, has) in candidates.items():
    tr, va = is_train & has, is_val & has
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1))
    clf.fit(X[tr], y[tr])
    pred = clf.predict(X[va])
    rows.append({"features": name, "dims": X.shape[1],
                 "val accuracy": (pred == y[va]).mean(),
                 "val macro F1": f1_score(y[va], pred, average="macro")})
check = pd.DataFrame(rows).set_index("features").round(3)
print(check)
print("\nFor comparison, TF-IDF text in notebook 02 got val macro F1 0.614")
try:
    check.to_csv(RESULTS_DIR / "feature_check.csv")
except OSError as e:
    print(f"Couldn't save feature_check.csv: {e}")

             dims  val accuracy  val macro F1
features                                     
effnet_crop  1280         0.490         0.490
effnet_pad   1280         0.484         0.483
resnet_pad   2048         0.473         0.474
text_bert     768         0.561         0.559

For comparison, TF-IDF text in notebook 02 got val macro F1 0.614


## Making sure everything is saved

Colab uploads to Drive in the background, so this waits until the feature files are actually on Drive before the runtime closes.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("Everything saved to Drive")

Everything saved to Drive
